# Stored execution evidence — Global Wheat Detection

This public evidence copy preserves every textual output cell supplied in `jiaozi.zip`.
The original notebook SHA-256 is `e298c0b6773ddde3a77045dc66d0da8a9071d9350c7b7ecff80755ce8b01db6c`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Global Wheat Detection Agent Notebook — Conditional YOLOv8 Execution Patch

This notebook keeps the original Jiaozi core:

`QUERY -> Module 1 -> Module 2 -> Module 3 -> Module 4`

Execution is dispatched according to the model selected by Jiaozi:

- If Jiaozi selects a YOLOv8 backbone, the temporary Ultralytics adapter maps the selected YOLOv8 scale to the corresponding public checkpoint, converts Global Wheat annotations to YOLO format, trains the selected detector, evaluates it, and writes the Kaggle submission.
- If Jiaozi selects any other backbone, the notebook keeps the original generated `run.py` / `train.py` execution path instead of raising an error.

The patch changes only the execution backend needed for YOLOv8; it does not override the agent's model choice.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or "").strip()
    markdown_match = re.fullmatch(r"\[(.*?)\]\((https?://[^)]+)\)", value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r"https?://\S+", value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value

# IMPORTANT: leave /content/Jiaozi before deleting it
os.chdir("/content")

REPO_URL = normalize_repo_url(
    os.getenv("JIAOZI_REPO_URL", "https://github.com/Isso-W/Jiaozi.git")
)
REPO_REF = os.getenv("JIAOZI_REPO_REF", "main")
REPO_DIR = Path("/content/Jiaozi")

print("Current working directory:", Path.cwd())
print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)
print("REPO_DIR =", REPO_DIR)

if REPO_DIR.exists():
    print("Removing old repo:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

print("Running:", " ".join(clone_cmd))

completed = subprocess.run(
    clone_cmd,
    cwd="/content",
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    timeout=300,
)

print("=== git stdout ===")
print(completed.stdout)

print("=== git stderr ===")
print(completed.stderr)

completed.check_returncode()

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo cloned successfully.")
print("Current working directory:", Path.cwd())
print("Top-level files:")
for p in sorted(REPO_DIR.iterdir()):
    print(" -", p.name)


Current working directory: /content
REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
REPO_DIR = /content/Jiaozi
Running: git clone --depth 1 --branch rag_wang https://github.com/muzhi-hac/Jiaozi-rag-wang.git /content/Jiaozi
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...

Repo cloned successfully.
Current working directory: /content/Jiaozi
Top-level files:
 - .env.example
 - .git
 - .gitignore
 - CLAUDE.md
 - LICENSE
 - README.md
 - analyzer
 - build_vision_benchmarks_notebook.py
 - configs
 - cost_meter.py
 - dataset_analyzer.py
 - docs
 - env_loader.py
 - features
 - features_extraction_api.py
 - ingestion
 - integration_update_colab.ipynb
 - kaggle_benchmark_colab.ipynb
 - kaggle_submit.py
 - main.py
 - module4_agent
 - pipeline.py
 - processors
 - pyproject.toml
 - recommender
 - report.md
 - requirements.txt
 - retrieval
 - run_and_log.py
 - run_for_testing.py
 - run_kaggle_benchmark.py
 - skrub_pipeline.py
 - test_pipeline.p

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

hf_token = read_secret('HF_TOKEN', required=False)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("[Cell 3] HF_TOKEN configured and Hugging Face login succeeded.")
    except Exception as exc:
        print("[Cell 3] HF_TOKEN was set, but huggingface_hub login failed:")
        print(repr(exc))
else:
    print("[Cell 3] No HF_TOKEN found.")

del openai_api_key
del kaggle_token
del hf_token



Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token
[Cell 3] No HF_TOKEN found.


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

#
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

#
kaggle_install = run_pip(["install", "--upgrade", "kaggle", "pycocotools"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

#  Kaggle access token
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# check kaggle availbility
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle pycocotools
==

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
KAGGLE_COMPETITION = 'global-wheat-detection'
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data


100% 607M/607M [00:30<00:00, 21.0MB/s]



In [5]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data")

zip_path = zip_files[0]
KAGGLE_DATA_DIR = DATA_ROOT / zip_path.stem
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(KAGGLE_DATA_DIR)

# Some Kaggle competitions contain nested archives. Extract them in place so image
# discovery below can find the real files regardless of packaging.
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as z:
        z.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for p in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", p.name)


Zip files: [PosixPath('/content/data/global-wheat-detection.zip')]
Extracting: /content/data/global-wheat-detection.zip
To: /content/data/global-wheat-detection
Done.
KAGGLE_DATA_DIR = /content/data/global-wheat-detection
Top-level files:
 - sample_submission.csv
 - test
 - train
 - train.csv


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated Jiaozi pipeline

This section:

1. Reads Global Wheat Detection files and converts the bbox table into Jiaozi-compatible metadata.
2. Runs the natural-language `QUERY` through Module 1.
3. Runs Module 2 on sampled real Kaggle images.
4. Runs Module 3 retrieval and recommender re-ranking.
5. Runs Module 4 code generation.
6. Saves the exact agent-selected configuration to `agent_selected_config.json`.

The following temporary adapter is used only when the selected backbone is YOLOv8, because the current generated loader treats `ultralytics/assets` as a Hugging Face checkpoint and otherwise falls back to `TinyBackbone`.


In [7]:
!pip -q install chromadb sentence-transformers networkx openai "datasets==3.6.0" pillow scikit-learn pandas matplotlib tqdm ultralytics pycocotools


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [8]:

import ast, json, os, shutil, sys
from pathlib import Path
import pandas as pd

REPO_DIR = Path('/content/Jiaozi')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('=' * 90)
print('[Global Wheat] keep Jiaozi core: QUERY -> Module1 -> Module2 -> Module3 -> Module4')

QUERY = (
    "Task: Global Wheat Detection, dense object detection for wheat head detection in field images. "
    "Dataset: Kaggle Global Wheat Detection. "
    "Input images are RGB wheat field images. Each image can contain many wheat heads. "
    "Training labels are bounding boxes, not image-level classes. "
    "The train.csv file contains image_id and bbox columns, where bbox is in [x, y, width, height] format. "
    "This is an object detection task, not a classification task. "
    "The agent should autonomously select the detection model, preprocessing, augmentation, optimizer, loss, and evaluation pipeline. "
    "The evaluation metric is mean average precision over IoU thresholds from 0.50 to 0.75, i.e. mAP@0.50:0.75. "
    "Do not use validation loss as the final evaluation metric. "
    "The generated pipeline should report local validation mAP@0.50:0.75 and use it to select the best checkpoint. "
    "The Kaggle submission format must contain image_id and PredictionString, where PredictionString is a sequence of score x y width height values."
)

print('QUERY passed to Jiaozi:')
print(QUERY)

REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path('/content/jiaozi_generated_pipeline')
DATASET_DIR = Path(KAGGLE_DATA_DIR)

train_csv = DATASET_DIR / 'train.csv'
sample_submission = DATASET_DIR / 'sample_submission.csv'
train_image_dir = DATASET_DIR / 'train'
test_image_dir = DATASET_DIR / 'test'
if not train_csv.exists():
    matches = list(DATASET_DIR.rglob('train.csv'))
    if matches:
        train_csv = matches[0]
        DATASET_DIR = train_csv.parent
        sample_submission = DATASET_DIR / 'sample_submission.csv'
        train_image_dir = DATASET_DIR / 'train'
        test_image_dir = DATASET_DIR / 'test'
for p in [train_csv, sample_submission, train_image_dir, test_image_dir]:
    print('check:', p, 'exists=', p.exists())
    if not p.exists(): raise FileNotFoundError(p)

raw = pd.read_csv(train_csv)
print('train.csv shape:', raw.shape)
print('columns:', list(raw.columns))
print(raw.head())

def parse_bbox(v):
    if isinstance(v, str): v = ast.literal_eval(v)
    x, y, w, h = [float(a) for a in v]
    return x, y, w, h

rows=[]
for _, r in raw.iterrows():
    x,y,w,h = parse_bbox(r['bbox'])
    if w <= 0 or h <= 0: continue
    image_id = str(r['image_id'])
    rows.append({'image_id':image_id, 'image':image_id+'.jpg', 'x':x, 'y':y, 'w':w, 'h':h, 'x1':x, 'y1':y, 'x2':x+w, 'y2':y+h, 'label':1})
bbox_frame=pd.DataFrame(rows)
processed_bbox_csv = DATASET_DIR / 'jiaozi_global_wheat_bboxes.csv'
bbox_frame.to_csv(processed_bbox_csv, index=False)
image_level_frame = bbox_frame.groupby(['image_id','image'], as_index=False).agg(num_boxes=('label','size'))
image_level_frame['label']=1
processed_image_csv = DATASET_DIR / 'jiaozi_global_wheat_image_level_for_module2.csv'
image_level_frame.to_csv(processed_image_csv, index=False)
print('processed_bbox_csv:', processed_bbox_csv)
print('bbox rows:', len(bbox_frame), 'unique images:', bbox_frame['image_id'].nunique())
print(image_level_frame['num_boxes'].describe())

LOCAL_DATA_INFO = {
    'benchmark':'global_wheat_detection','competition':'global-wheat-detection','task_type':'object_detection',
    'train_csv':str(processed_bbox_csv),'image_level_csv':str(processed_image_csv),'image_dir':str(train_image_dir),
    'test_image_dir':str(test_image_dir),'sample_submission':str(sample_submission),'image_column':'image','image_id_column':'image_id',
    'bbox_format':'xywh','detector_bbox_format':'xyxy','num_object_classes':1,'num_classes':2,
    'class_names':['background','wheat_head'],'metric':'detection_map','submission_format':'image_id,PredictionString'
}
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import build_all_task_lists, build_graph, build_vector_index, print_results, retrieve_top3_hybrid

print('\n[Jiaozi Module 1]')
m1_output = module1_pipeline(QUERY)
if m1_output is None: raise RuntimeError('Module 1 failed')
print(json.dumps(m1_output, indent=2, ensure_ascii=False, default=str)[:3000])

class LocalImageSplit:
    column_names=['image','label']
    def __init__(self, frame, info):
        from PIL import Image
        self._Image=Image; self.labels=frame['label'].tolist(); root=Path(info['image_dir'])
        self.image_paths=[root/str(v) for v in frame['image'].tolist()]
    def __len__(self): return len(self.labels)
    def __getitem__(self,key):
        if key=='label': return self.labels
        if key=='image': return [self._open(p) for p in self.image_paths[:200]]
        i=int(key); return {'image':self._open(self.image_paths[i]), 'label':self.labels[i]}
    def _open(self,p):
        with self._Image.open(p) as im: return im.convert('RGB')

def build_local_module2_dataset(info):
    f=pd.read_csv(info['image_level_csv'])
    if len(f)>200: f=f.sample(n=200, random_state=42).reset_index(drop=True)
    return {'train':LocalImageSplit(f, info)}

print('\n[Jiaozi Module 2]')
m2_report = ImageStatisticsAnalyzer().analyze(build_local_module2_dataset(LOCAL_DATA_INFO))
print(json.dumps(m2_report, indent=2, ensure_ascii=False, default=str)[:3000])

m3_input = merge_modules(m1_output, m2_report)
m3_input.update({'task_type':'object_detection','problem_type':'dense_object_detection','evaluation_metric':'detection_map','num_classes':2,'num_object_classes':1,'label_type':'bounding_boxes','bbox_format':'xywh','submission_format':'image_id,PredictionString'})
print('\n[Merged input to Module 3]')
print(json.dumps(m3_input, indent=2, ensure_ascii=False, default=str)[:3000])

print('\n[Jiaozi Module 3]')
graph=build_graph(); collection=build_vector_index(); recommendations=retrieve_top3_hybrid(m3_input, graph, collection)
print_results(m3_input, recommendations, graph)
try:
    recommendations = recommend(recommendations, m2_report, m3_input, memory=OutcomeMemory())
    print('Recommender re-ranked candidates')
except Exception as e:
    print('Recommender skipped:', e)

task_lists = build_all_task_lists(recommendations, graph, fmt='nl')
if not task_lists: raise RuntimeError('Jiaozi returned no task_lists')
for tl in task_lists:
    mc=tl.get('model_config')
    if isinstance(mc, dict):
        mc.update({
    **LOCAL_DATA_INFO,
    "offline_smoke": not REAL_TRAINING,
    "evaluation_metric": "mAP@0.50:0.75",
})
        mc.setdefault('recommended_epochs', derive_recommended_epochs(m3_input.get('data_size','medium'), mc.get('finetune_strategy'), bool(mc.get('pretrained_hf_id'))))

agent_top_config = task_lists[0].get('model_config', {})
print('\n[Jiaozi selected top model_config]')
print(json.dumps(agent_top_config, indent=2, ensure_ascii=False, default=str)[:5000])



print('\n[Jiaozi Module 4]')
module4 = run_module4_generation(task_lists, OUTPUT_DIR, skip_smoke=REAL_TRAINING, timeout=120, llm_provider='openai')
summary_path=OUTPUT_DIR/'module4_summary.json'
if not summary_path.exists(): raise FileNotFoundError(summary_path)
print(summary_path.read_text(encoding='utf-8')[:3000])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR/'agent_selected_config.json').write_text(
    json.dumps(agent_top_config, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8'
)
DATASET = LOCAL_DATA_INFO['train_csv']
print('Jiaozi complete. Next cells write detection adapter files and train.')


[Global Wheat] keep Jiaozi core: QUERY -> Module1 -> Module2 -> Module3 -> Module4
QUERY passed to Jiaozi:
Task: Global Wheat Detection, dense object detection for wheat head detection in field images. Dataset: Kaggle Global Wheat Detection. Input images are RGB wheat field images. Each image can contain many wheat heads. Training labels are bounding boxes, not image-level classes. The train.csv file contains image_id and bbox columns, where bbox is in [x, y, width, height] format. This is an object detection task, not a classification task. The agent should autonomously select the detection model, preprocessing, augmentation, optimizer, loss, and evaluation pipeline. The evaluation metric is mean average precision over IoU thresholds from 0.50 to 0.75, i.e. mAP@0.50:0.75. Do not use validation loss as the final evaluation metric. The generated pipeline should report local validation mAP@0.50:0.75 and use it to select the best checkpoint. The Kaggle submission format must contain image

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : object_detection
Input  : size=small  priority=balanced
Desc   : Task: Global Wheat Detection, dense object detection for wheat head detection in field images. Dataset: Kaggle Global Wheat Detection. Input images are RGB wheat field images. Each image can contain many wheat heads. Training labels are bounding boxes, not image-level classes. The train.csv file contains image_id and bbox columns, where bbox is in [x, y, width, height] format. This is an object detection task, not a classification task. The agent should autonomously select the detection model, preprocessing, augmentation, optimizer, loss, and evaluation pipeline. The evaluation metric is mean average precision over IoU thresholds from 0.50 to 0.75, i.e. mAP@0.50:0.75. Do not use validation loss as the final evaluation metric. The generated pipeline should report local validatio

In [9]:
from pathlib import Path
for name in ['configs.json','agent_selected_config.json','model.py','train.py','run.py']:
    p=Path('/content/jiaozi_generated_pipeline')/name
    print(name, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else None)


configs.json exists= True size= 8093
agent_selected_config.json exists= True size= 1330
model.py exists= True size= 8865
train.py exists= True size= 47764
run.py exists= True size= 4665


In [10]:
import json
from pathlib import Path

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")

for name in [
    "configs.json",
    "agent_selected_config.json",
    "module3_candidates.json",
    "generation_info.json",
    "module4_summary.json",
]:
    path = OUTPUT_DIR / name
    if not path.exists():
        continue

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    obj = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))



configs.json
[
  {
    "alternatives": [
      "yolo26"
    ],
    "augmentation": "basic",
    "backbone": "yolov8",
    "checkpoint": "ultralytics/assets",
    "class_imbalance": false,
    "data_size": "medium",
    "embedding_dim": 32,
    "finetune_strategy": "full",
    "freeze_backbone": false,
    "head": "detection_head_anchor_free",
    "image_size": 224,
    "learning_rate": 0.001,
    "loss": "focal_loss",
    "model_config": {
      "backbone": "yolov8",
      "bbox_format": "xywh",
      "benchmark": "global_wheat_detection",
      "class_names": [
        "background",
        "wheat_head"
      ],
      "competition": "global-wheat-detection",
      "detector_bbox_format": "xyxy",
      "evaluation_metric": "mAP@0.50:0.75",
      "finetune_strategy": "full",
      "freeze_backbone": false,
      "head": "detection_head_anchor_free",
      "image_column": "image",
      "image_dir": "/content/data/global-wheat-detection/train",
      "image_id_column": "image_id",
     

## Conditional execution dispatcher

This section reads `agent_selected_config.json` and chooses the execution path:

- `YOLOv8` → use the temporary Ultralytics execution adapter;
- any other selected model → run the original Jiaozi-generated `run.py` or `train.py`.

Therefore, a non-YOLO recommendation is not treated as an error and is not overridden.


In [12]:
import json
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
AGENT_CONFIG_PATH = OUTPUT_DIR / "agent_selected_config.json"

if not AGENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "agent_selected_config.json is missing. Run the Jiaozi Module 1-4 cell first."
    )

agent_cfg = json.loads(AGENT_CONFIG_PATH.read_text(encoding="utf-8"))

selected_backbone = str(agent_cfg.get("backbone", "")).strip().lower()
selected_name = str(agent_cfg.get("pretrained_name", "")).strip()
selected_checkpoint = str(
    agent_cfg.get("pretrained_checkpoint")
    or agent_cfg.get("checkpoint")
    or agent_cfg.get("pretrained_hf_id")
    or ""
).strip()
recommended_epochs = int(agent_cfg.get("recommended_epochs", 40))

USE_YOLOV8_PATCH = selected_backbone.startswith("yolov8")

print("=" * 90)
print("[Conditional execution dispatcher]")
print("Agent-selected backbone       :", selected_backbone)
print("Agent-selected pretrained name:", selected_name)
print("Agent-selected checkpoint     :", selected_checkpoint)
print("Agent-recommended epochs      :", recommended_epochs)
print("Agent target metric           :", agent_cfg.get("evaluation_metric"))
print("Use temporary YOLOv8 patch    :", USE_YOLOV8_PATCH)
print("=" * 90)

if not USE_YOLOV8_PATCH:
    print(
        "The agent selected a non-YOLOv8 model. "
        "Running the original Jiaozi-generated execution path."
    )

    run_path = OUTPUT_DIR / "run.py"
    train_path = OUTPUT_DIR / "train.py"

    if run_path.exists():
        command = [
            sys.executable,
            "-u",
            str(run_path),
            "--epochs",
            str(recommended_epochs),
        ]
    elif train_path.exists():
        command = [
            sys.executable,
            "-u",
            str(train_path),
            "--epochs",
            str(recommended_epochs),
        ]
    else:
        raise FileNotFoundError(
            "Jiaozi did not generate run.py or train.py for the selected model."
        )

    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=str(OUTPUT_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Original Jiaozi-generated pipeline failed with return code {return_code}."
        )

    print("Original Jiaozi-generated execution completed successfully.")

else:
    def resolve_ultralytics_checkpoint(config: dict) -> str:
        """Map the agent-selected YOLOv8 scale to an executable Ultralytics checkpoint."""
        explicit = str(
            config.get("pretrained_checkpoint")
            or config.get("checkpoint")
            or ""
        ).strip()

        if explicit.lower().endswith(".pt") and "yolov8" in explicit.lower():
            return explicit

        name = str(config.get("pretrained_name", "")).lower()
        backbone = str(config.get("backbone", "")).lower()
        params_m = float(config.get("params_M", 0) or 0)

        if any(token in name for token in ["xlarge", "extra large", "yolov8x"]) or "yolov8x" in backbone:
            return "yolov8x.pt"
        if any(token in name for token in ["large", "yolov8l"]) or "yolov8l" in backbone:
            return "yolov8l.pt"
        if any(token in name for token in ["medium", "yolov8m"]) or "yolov8m" in backbone:
            return "yolov8m.pt"
        if any(token in name for token in ["small", "yolov8s"]) or "yolov8s" in backbone:
            return "yolov8s.pt"
        if any(token in name for token in ["nano", "yolov8n"]) or "yolov8n" in backbone:
            return "yolov8n.pt"

        # Parameter-based fallback preserves the agent's intended model scale.
        if params_m >= 60:
            return "yolov8x.pt"
        if params_m >= 35:
            return "yolov8l.pt"
        if params_m >= 15:
            return "yolov8m.pt"
        if params_m >= 5:
            return "yolov8s.pt"
        return "yolov8n.pt"


    YOLO_WEIGHTS = resolve_ultralytics_checkpoint(agent_cfg)
    print("Resolved executable checkpoint:", YOLO_WEIGHTS)

    # Runtime options
    SPLIT_SEED = 42
    VAL_FRACTION = 0.10
    IMAGE_SIZE = 1024
    BATCH_SIZE = 2
    NUM_WORKERS = 2
    REBUILD_YOLO_DATA = True
    RESUME_YOLO = False

    train_csv = Path(agent_cfg["train_csv"])
    train_image_dir = Path(agent_cfg["image_dir"])
    test_image_dir = Path(agent_cfg["test_image_dir"])
    sample_submission_path = Path(agent_cfg["sample_submission"])

    for label, path in {
        "train_csv": train_csv,
        "train_image_dir": train_image_dir,
        "test_image_dir": test_image_dir,
        "sample_submission": sample_submission_path,
    }.items():
        print(f"{label:20}: {path} exists={path.exists()}")
        if not path.exists():
            raise FileNotFoundError(path)

    bbox_df = pd.read_csv(train_csv)

    required_columns = {"image_id", "x", "y", "w", "h"}
    missing_columns = required_columns - set(bbox_df.columns)
    if missing_columns:
        raise ValueError(
            f"Processed bbox CSV is missing columns: {sorted(missing_columns)}"
        )

    if "image" not in bbox_df.columns:
        bbox_df["image"] = bbox_df["image_id"].astype(str) + ".jpg"

    YOLO_DATA_DIR = OUTPUT_DIR / "yolo_adapter_data"
    YOLO_RUNS_DIR = OUTPUT_DIR / "yolo_runs"
    YOLO_RUN_NAME = "agent_selected_yolov8_global_wheat"
    YOLO_RUN_DIR = YOLO_RUNS_DIR / YOLO_RUN_NAME

    if REBUILD_YOLO_DATA and YOLO_DATA_DIR.exists():
        shutil.rmtree(YOLO_DATA_DIR)

    for split in ["train", "val"]:
        (YOLO_DATA_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DATA_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    image_ids = sorted(bbox_df["image_id"].astype(str).unique().tolist())
    rng = random.Random(SPLIT_SEED)
    rng.shuffle(image_ids)

    val_count = max(1, int(round(len(image_ids) * VAL_FRACTION)))
    val_ids = set(image_ids[:val_count])
    train_ids = set(image_ids[val_count:])

    if not train_ids or not val_ids:
        raise RuntimeError("The deterministic train/validation split is empty.")

    split_manifest = pd.DataFrame(
        {
            "image_id": image_ids,
            "split": ["val" if image_id in val_ids else "train" for image_id in image_ids],
        }
    )
    split_manifest_path = OUTPUT_DIR / "yolo_split_manifest.csv"
    split_manifest.to_csv(split_manifest_path, index=False)

    print("YOLO train images:", len(train_ids))
    print("YOLO val images  :", len(val_ids))
    print("Split manifest   :", split_manifest_path)


    def find_image(image_root: Path, image_name: str, image_id: str) -> Path:
        candidates = [
            image_root / image_name,
            image_root / f"{image_id}.jpg",
            image_root / f"{image_id}.jpeg",
            image_root / f"{image_id}.png",
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        matches = list(image_root.glob(f"{image_id}.*"))
        if matches:
            return matches[0]
        raise FileNotFoundError(f"No image found for image_id={image_id}")


    def link_or_copy(source: Path, destination: Path) -> None:
        if destination.exists() or destination.is_symlink():
            destination.unlink()
        try:
            os.symlink(source.resolve(), destination)
        except OSError:
            shutil.copy2(source, destination)


    written_images = 0
    written_boxes = 0

    for image_id, group in bbox_df.groupby(bbox_df["image_id"].astype(str), sort=False):
        split = "val" if image_id in val_ids else "train"
        image_name = str(group.iloc[0]["image"])
        source_image = find_image(train_image_dir, image_name, image_id)

        with Image.open(source_image) as image:
            image_width, image_height = image.size

        destination_image = YOLO_DATA_DIR / "images" / split / source_image.name
        link_or_copy(source_image, destination_image)

        label_lines = []
        for row in group.itertuples(index=False):
            x = float(row.x)
            y = float(row.y)
            w = float(row.w)
            h = float(row.h)

            if w <= 0 or h <= 0:
                continue

            x_center = (x + w / 2.0) / image_width
            y_center = (y + h / 2.0) / image_height
            width_norm = w / image_width
            height_norm = h / image_height

            # Clamp tiny annotation rounding errors into the valid YOLO interval.
            x_center = min(max(x_center, 0.0), 1.0)
            y_center = min(max(y_center, 0.0), 1.0)
            width_norm = min(max(width_norm, 0.0), 1.0)
            height_norm = min(max(height_norm, 0.0), 1.0)

            label_lines.append(
                f"0 {x_center:.8f} {y_center:.8f} {width_norm:.8f} {height_norm:.8f}"
            )

        label_path = (
            YOLO_DATA_DIR
            / "labels"
            / split
            / f"{destination_image.stem}.txt"
        )
        label_path.write_text("\n".join(label_lines), encoding="utf-8")

        written_images += 1
        written_boxes += len(label_lines)

    print("Prepared YOLO images:", written_images)
    print("Prepared YOLO boxes :", written_boxes)

    data_yaml_path = YOLO_DATA_DIR / "global_wheat.yaml"
    data_yaml_path.write_text(
        "\n".join(
            [
                f"path: {YOLO_DATA_DIR}",
                "train: images/train",
                "val: images/val",
                "nc: 1",
                "names:",
                "  0: wheat_head",
                "",
            ]
        ),
        encoding="utf-8",
    )

    print("YOLO data YAML:")
    print(data_yaml_path.read_text(encoding="utf-8"))

    device = 0 if torch.cuda.is_available() else "cpu"
    optimizer_name = (
        "SGD"
        if "sgd" in str(agent_cfg.get("optimizer", "")).lower()
        else "auto"
    )

    last_checkpoint = YOLO_RUN_DIR / "weights" / "last.pt"

    if RESUME_YOLO and last_checkpoint.exists():
        print("Resuming YOLO training from:", last_checkpoint)
        yolo_model = YOLO(str(last_checkpoint))
        train_result = yolo_model.train(resume=True)
    else:
        print("Loading agent-selected Ultralytics checkpoint:", YOLO_WEIGHTS)
        yolo_model = YOLO(YOLO_WEIGHTS)

        # Ultralytics prints one structured validation line after every epoch.
        train_result = yolo_model.train(
            data=str(data_yaml_path),
            epochs=recommended_epochs,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            workers=NUM_WORKERS,
            device=device,
            optimizer=optimizer_name,
            project=str(YOLO_RUNS_DIR),
            name=YOLO_RUN_NAME,
            exist_ok=True,
            seed=SPLIT_SEED,
            deterministic=True,
            pretrained=True,
            plots=True,
            verbose=True,
        )

    trainer_save_dir = getattr(getattr(yolo_model, "trainer", None), "save_dir", None)
    if trainer_save_dir is not None:
        YOLO_RUN_DIR = Path(trainer_save_dir)

    BEST_YOLO_PATH = YOLO_RUN_DIR / "weights" / "best.pt"
    LAST_YOLO_PATH = YOLO_RUN_DIR / "weights" / "last.pt"

    print("=" * 90)
    print("Temporary YOLOv8 adapter training completed.")
    print("Actual Ultralytics checkpoint:", YOLO_WEIGHTS)
    print("Run directory                :", YOLO_RUN_DIR)
    print("Best checkpoint              :", BEST_YOLO_PATH, "exists=", BEST_YOLO_PATH.exists())
    print("Last checkpoint              :", LAST_YOLO_PATH, "exists=", LAST_YOLO_PATH.exists())
    print("=" * 90)

    if not BEST_YOLO_PATH.exists():
        raise FileNotFoundError(
            "YOLOv8 training finished without best.pt. Check the Ultralytics logs above."
        )

    patch_state = {
        "adapter": "temporary_ultralytics_yolov8",
        "agent_selected_backbone": selected_backbone,
        "agent_selected_pretrained_name": selected_name,
        "agent_selected_checkpoint": selected_checkpoint,
        "resolved_ultralytics_checkpoint": YOLO_WEIGHTS,
        "recommended_epochs": recommended_epochs,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "split_seed": SPLIT_SEED,
        "val_fraction": VAL_FRACTION,
        "data_yaml": str(data_yaml_path),
        "split_manifest": str(split_manifest_path),
        "run_dir": str(YOLO_RUN_DIR),
        "best_checkpoint": str(BEST_YOLO_PATH),
        "last_checkpoint": str(LAST_YOLO_PATH),
    }
    (OUTPUT_DIR / "yolov8_patch_state.json").write_text(
        json.dumps(patch_state, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("Saved patch state:", OUTPUT_DIR / "yolov8_patch_state.json")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[Conditional execution dispatcher]
Agent-selected backbone       : yolov8
Agent-selected pretrained name: YOLOv8-Medium / COCO
Agent-selected checkpoint     : ultralytics/assets
Agent-recommended epochs      : 40
Agent target metric           : mAP@0.50:0.75
Use temporary YOLOv8 patch    : True
Resolved executable checkpoint: yolov8m.pt
train_csv           : /content/data/global-wheat-detection/jiaozi_global_wheat_bboxes.csv exists=True
train_image_dir     : /content/data/global-wheat-detection/train exists=True
test_image_dir      : /content/data/global-wheat-detection/test exists=True
sample_submission   : /content/data/global-wheat-detection/sample_submission.csv exists=True
YO

In [14]:
if not globals().get("USE_YOLOV8_PATCH", False):
    print(
        "Skipping the YOLOv8-specific evaluation/submission cell because "
        "the agent selected a different model. The original generated pipeline "
        "has already handled its own evaluation/inference path."
    )
else:
    import gc
    import json
    import math
    import shutil
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    from ultralytics import YOLO

    OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
    patch_state_path = OUTPUT_DIR / "yolov8_patch_state.json"
    agent_cfg_path = OUTPUT_DIR / "agent_selected_config.json"

    if not patch_state_path.exists():
        raise FileNotFoundError(
            "yolov8_patch_state.json is missing. Run the temporary YOLOv8 training cell first."
        )

    patch_state = json.loads(patch_state_path.read_text(encoding="utf-8"))
    agent_cfg = json.loads(agent_cfg_path.read_text(encoding="utf-8"))

    best_checkpoint = Path(patch_state["best_checkpoint"])
    split_manifest_path = Path(patch_state["split_manifest"])
    train_csv = Path(agent_cfg["train_csv"])
    train_image_dir = Path(agent_cfg["image_dir"])
    test_image_dir = Path(agent_cfg["test_image_dir"])
    sample_submission_path = Path(agent_cfg["sample_submission"])

    for path in [
        best_checkpoint,
        split_manifest_path,
        train_csv,
        train_image_dir,
        test_image_dir,
        sample_submission_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    # Release the training model retained by the previous cell before inference.
    # The previous version kept both the training model and a second best.pt model
    # on the GPU, which could exhaust the 24 GB L4 memory.
    for variable_name in ["yolo_model", "train_result", "model"]:
        globals().pop(variable_name, None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    inference_device = 0 if torch.cuda.is_available() else "cpu"
    use_half_precision = bool(torch.cuda.is_available())

    print("Inference device :", inference_device)
    print("Half precision   :", use_half_precision)

    model = YOLO(str(best_checkpoint))
    bbox_df = pd.read_csv(train_csv)
    split_manifest = pd.read_csv(split_manifest_path)

    val_ids = set(
        split_manifest.loc[
            split_manifest["split"].eq("val"),
            "image_id",
        ].astype(str)
    )
    val_frame = bbox_df[bbox_df["image_id"].astype(str).isin(val_ids)].copy()

    CONF_THRESHOLD = 0.05
    NMS_IOU_THRESHOLD = 0.60
    MAX_DETECTIONS = 300
    IMAGE_SIZE = int(patch_state["image_size"])
    IOU_THRESHOLDS = np.arange(0.50, 0.76, 0.05)


    def find_image(image_root: Path, image_name: str, image_id: str) -> Path:
        candidates = [
            image_root / image_name,
            image_root / f"{image_id}.jpg",
            image_root / f"{image_id}.jpeg",
            image_root / f"{image_id}.png",
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        matches = list(image_root.glob(f"{image_id}.*"))
        if matches:
            return matches[0]
        raise FileNotFoundError(f"No image found for image_id={image_id}")


    def pairwise_iou(box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
        if boxes.size == 0:
            return np.empty((0,), dtype=np.float32)

        x1 = np.maximum(box[0], boxes[:, 0])
        y1 = np.maximum(box[1], boxes[:, 1])
        x2 = np.minimum(box[2], boxes[:, 2])
        y2 = np.minimum(box[3], boxes[:, 3])

        intersection = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
        box_area = max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])
        boxes_area = (
            np.maximum(0.0, boxes[:, 2] - boxes[:, 0])
            * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])
        )
        union = box_area + boxes_area - intersection

        return np.divide(
            intersection,
            union,
            out=np.zeros_like(intersection, dtype=np.float32),
            where=union > 0,
        )


    def image_precision(
        predicted_boxes: np.ndarray,
        predicted_scores: np.ndarray,
        ground_truth_boxes: np.ndarray,
        iou_threshold: float,
    ) -> float:
        """Global-Wheat-style image precision: TP / (TP + FP + FN)."""
        if len(predicted_boxes) == 0 and len(ground_truth_boxes) == 0:
            return 1.0
        if len(predicted_boxes) == 0 or len(ground_truth_boxes) == 0:
            return 0.0

        order = np.argsort(-predicted_scores)
        predicted_boxes = predicted_boxes[order]

        matched_ground_truth = np.zeros(len(ground_truth_boxes), dtype=bool)
        true_positives = 0
        false_positives = 0

        for predicted_box in predicted_boxes:
            available_indices = np.where(~matched_ground_truth)[0]
            if len(available_indices) == 0:
                false_positives += 1
                continue

            overlaps = pairwise_iou(
                predicted_box,
                ground_truth_boxes[available_indices],
            )
            best_local_index = int(np.argmax(overlaps))
            best_iou = float(overlaps[best_local_index])

            if best_iou >= iou_threshold:
                true_positives += 1
                matched_ground_truth[available_indices[best_local_index]] = True
            else:
                false_positives += 1

        false_negatives = int((~matched_ground_truth).sum())
        denominator = true_positives + false_positives + false_negatives

        return true_positives / denominator if denominator else 0.0


    print("=" * 90)
    print("[Local validation: Global Wheat mAP@0.50:0.75 proxy]")
    print("Best checkpoint :", best_checkpoint)
    print("Validation images:", len(val_ids))
    print("Confidence      :", CONF_THRESHOLD)
    print("NMS IoU         :", NMS_IOU_THRESHOLD)
    print("=" * 90)

    val_image_paths = []
    for image_id, group in val_frame.groupby(val_frame["image_id"].astype(str), sort=False):
        image_name = (
            str(group.iloc[0]["image"])
            if "image" in group.columns
            else f"{image_id}.jpg"
        )
        val_image_paths.append(
            find_image(train_image_dir, image_name, image_id)
        )

    prediction_by_image_id = {}

    # Predict one image at a time. Passing the full list of 337 validation images
    # can make Ultralytics construct a very large inference batch and cause CUDA OOM.
    for index, image_path in enumerate(val_image_paths, start=1):
        result = model.predict(
            source=str(image_path),
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=NMS_IOU_THRESHOLD,
            max_det=MAX_DETECTIONS,
            device=inference_device,
            half=use_half_precision,
            verbose=False,
        )[0]

        image_id = Path(result.path).stem

        if result.boxes is None or len(result.boxes) == 0:
            boxes = np.empty((0, 4), dtype=np.float32)
            scores = np.empty((0,), dtype=np.float32)
        else:
            boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
            scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)

        prediction_by_image_id[image_id] = (boxes, scores)

        if index == 1 or index % 25 == 0 or index == len(val_image_paths):
            print(
                f"[validation inference] {index}/{len(val_image_paths)} "
                f"image_id={image_id} boxes={len(boxes)}",
                flush=True,
            )

        del result
        if torch.cuda.is_available() and index % 50 == 0:
            torch.cuda.empty_cache()

    image_metric_values = []

    for image_id, group in val_frame.groupby(val_frame["image_id"].astype(str), sort=False):
        gt_boxes = np.column_stack(
            [
                group["x"].to_numpy(dtype=np.float32),
                group["y"].to_numpy(dtype=np.float32),
                group["x"].to_numpy(dtype=np.float32)
                + group["w"].to_numpy(dtype=np.float32),
                group["y"].to_numpy(dtype=np.float32)
                + group["h"].to_numpy(dtype=np.float32),
            ]
        )

        predicted_boxes, predicted_scores = prediction_by_image_id.get(
            image_id,
            (
                np.empty((0, 4), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            ),
        )

        threshold_values = [
            image_precision(
                predicted_boxes,
                predicted_scores,
                gt_boxes,
                float(iou_threshold),
            )
            for iou_threshold in IOU_THRESHOLDS
        ]
        image_metric_values.append(float(np.mean(threshold_values)))

    local_map_050_075 = float(np.mean(image_metric_values))

    print(f"Local validation mAP@0.50:0.75 proxy = {local_map_050_075:.6f}")

    # ------------------------------------------------------------------
    # Generate Kaggle submission with the same trained best.pt checkpoint.
    # ------------------------------------------------------------------
    sample_submission = pd.read_csv(sample_submission_path)

    expected_columns = ["image_id", "PredictionString"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(
            "Unexpected sample submission columns: "
            f"{list(sample_submission.columns)}"
        )

    test_lookup = {
        path.stem: path
        for path in test_image_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    }

    ordered_test_paths = []
    for image_id in sample_submission["image_id"].astype(str):
        if image_id not in test_lookup:
            raise FileNotFoundError(
                f"Test image not found for sample image_id={image_id}"
            )
        ordered_test_paths.append(test_lookup[image_id])

    test_predictions = {}

    for index, image_path in enumerate(ordered_test_paths, start=1):
        result = model.predict(
            source=str(image_path),
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=NMS_IOU_THRESHOLD,
            max_det=MAX_DETECTIONS,
            device=inference_device,
            half=use_half_precision,
            verbose=False,
        )[0]

        image_id = Path(result.path).stem
        prediction_parts = []

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.detach().cpu().numpy()
            scores = result.boxes.conf.detach().cpu().numpy()

            order = np.argsort(-scores)
            for box, score in zip(boxes[order], scores[order]):
                x1, y1, x2, y2 = [float(value) for value in box]
                width = max(0.0, x2 - x1)
                height = max(0.0, y2 - y1)

                if width <= 0 or height <= 0:
                    continue

                prediction_parts.extend(
                    [
                        f"{float(score):.6f}",
                        f"{x1:.2f}",
                        f"{y1:.2f}",
                        f"{width:.2f}",
                        f"{height:.2f}",
                    ]
                )

        test_predictions[image_id] = " ".join(prediction_parts)

        if index == 1 or index % 25 == 0 or index == len(ordered_test_paths):
            print(
                f"[test inference] {index}/{len(ordered_test_paths)} "
                f"image_id={image_id} boxes={len(prediction_parts) // 5}",
                flush=True,
            )

        del result
        if torch.cuda.is_available() and index % 50 == 0:
            torch.cuda.empty_cache()

    submission = sample_submission[["image_id"]].copy()
    submission["PredictionString"] = (
        submission["image_id"].astype(str).map(test_predictions).fillna("")
    )

    submission_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False)

    result_summary = {
        **patch_state,
        "local_metric_name": "global_wheat_map_0.50_0.75_proxy",
        "local_metric_value": local_map_050_075,
        "confidence_threshold": CONF_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "validation_images": len(val_ids),
        "submission_path": str(submission_path),
        "submission_rows": len(submission),
    }

    result_summary_path = OUTPUT_DIR / "yolov8_patch_results.json"
    result_summary_path.write_text(
        json.dumps(result_summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("=" * 90)
    print("Temporary YOLOv8 adapter result")
    print("Actual model checkpoint :", patch_state["resolved_ultralytics_checkpoint"])
    print("Best trained checkpoint :", best_checkpoint)
    print("Local metric            :", f"{local_map_050_075:.6f}")
    print("Submission              :", submission_path)
    print("Result summary          :", result_summary_path)
    print("=" * 90)

    # Save only compact artifacts to Drive. The full local dataset is not copied.
    drive_backup_dir = Path(
        "/content/drive/MyDrive/Jiaozi/global_wheat_yolov8_patch"
    )
    if Path("/content/drive/MyDrive").exists():
        drive_backup_dir.mkdir(parents=True, exist_ok=True)

        for source in [
            best_checkpoint,
            submission_path,
            result_summary_path,
            patch_state_path,
            split_manifest_path,
        ]:
            if source.exists():
                destination = drive_backup_dir / source.name
                shutil.copy2(source, destination)
                print("Backed up:", destination)
    else:
        print("Google Drive is not mounted; compact artifact backup skipped.")


Inference device : 0
Half precision   : True
[Local validation: Global Wheat mAP@0.50:0.75 proxy]
Best checkpoint : /content/jiaozi_generated_pipeline/yolo_runs/agent_selected_yolov8_global_wheat/weights/best.pt
Validation images: 337
Confidence      : 0.05
NMS IoU         : 0.6
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
[validation inference] 1/337 image_id=b53afdf5c boxes=47
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is depre

In [15]:
import json
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
sub_path = OUTPUT_DIR / "submission.csv"

agent_cfg = json.loads(
    (OUTPUT_DIR / "agent_selected_config.json").read_text(encoding="utf-8")
)
sample_path = Path(agent_cfg["sample_submission"])

print("submission exists:", sub_path.exists(), sub_path)
print("sample exists:", sample_path.exists(), sample_path)

if not sub_path.exists():
    raise FileNotFoundError(
        "submission.csv is missing. Run the YOLOv8 evaluation/submission cell first."
    )

sub = pd.read_csv(sub_path)
sample = pd.read_csv(sample_path)

print("Submission shape:", sub.shape)
print("Sample shape:", sample.shape)
print("Submission columns:", list(sub.columns))
print("Sample columns:", list(sample.columns))

print("\nSubmission head:")
print(sub.head(10))

print("\nNaN counts before fill:")
print(sub.isna().sum())

assert list(sub.columns) == ["image_id", "PredictionString"], (
    "Columns must be image_id, PredictionString"
)
assert sub.shape[0] == sample.shape[0], (
    "Row count must match sample_submission"
)
assert set(sub["image_id"].astype(str)) == set(
    sample["image_id"].astype(str)
), "image_id set must match sample_submission"

sub["PredictionString"] = sub["PredictionString"].fillna("").astype(str)
sub.to_csv(sub_path, index=False)

print("\nPredictionString examples:")
for _, row in sub.head(10).iterrows():
    print(
        row["image_id"],
        "=>",
        repr(row["PredictionString"][:200]),
    )

print("\nBasic submission format check passed.")


submission exists: True /content/jiaozi_generated_pipeline/submission.csv
sample exists: True /content/data/global-wheat-detection/sample_submission.csv
Submission shape: (10, 2)
Sample shape: (10, 2)
Submission columns: ['image_id', 'PredictionString']
Sample columns: ['image_id', 'PredictionString']

Submission head:
    image_id                                   PredictionString
0  aac893a91  0.834473 238.00 79.19 140.50 146.56 0.817383 5...
1  51f1be19e  0.796875 602.00 80.56 170.00 179.44 0.782715 5...
2  f5a1f0358  0.841309 942.00 430.00 82.00 187.00 0.806641 5...
3  796707dd7  0.803711 712.50 826.00 105.00 100.00 0.787109 ...
4  51b3e36ab  0.823242 230.25 640.00 97.50 160.00 0.812500 8...
5  348a992bb  0.811523 0.31 316.00 123.25 98.00 0.810059 761...
6  cc3532ff6  0.806641 772.50 824.00 164.00 160.00 0.803223 ...
7  2fd875eaa  0.855957 890.00 52.25 104.00 87.00 0.844727 10...
8  cb8d261a3  0.828613 650.00 681.50 94.00 67.00 0.813965 75...
9  53f253011  0.830078 621.00 102.31 11

In [16]:
# import pandas as pd
# from pathlib import Path

# sample_path = Path("/content/data/global-wheat-detection/sample_submission.csv")
# sample = pd.read_csv(sample_path)

# test_sub = sample.copy()
# test_sub["PredictionString"] = "1.0 0 0 50 50"

# test_path = Path("/content/jiaozi_generated_pipeline/test_sample_submission.csv")
# test_sub.to_csv(test_path, index=False)

# print(test_path)
# print(test_sub.head())


In [19]:
!kaggle competitions submit \
  -c global-wheat-detection \
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m 'Jiaozi Global Wheat Detection agent bbox submission'


100% 12.4k/12.4k [00:00<00:00, 15.3kB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


In [18]:
!kaggle competitions submissions -c global-wheat-detection


No submissions found


In [20]:
from google.colab import files
files.download("/content/jiaozi_generated_pipeline/yolo_runs/agent_selected_yolov8_global_wheat/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from pathlib import Path
import shutil
from google.colab import files

pipeline_dir = Path("/content/jiaozi_generated_pipeline")

if not pipeline_dir.is_dir():
    raise FileNotFoundError(f"Directory does not exist：{pipeline_dir}")

# Search for all PyTorch weights such as best.pt, best_model.pt, and last.pt.
weight_files = sorted(pipeline_dir.rglob("*.pt"))

if not weight_files:
    raise FileNotFoundError(
        f"exist {pipeline_dir} No .pt weight files were found."
    )

print("Find the following weight file：")
for path in weight_files:
    print(
        f"  {path.relative_to(pipeline_dir)}"
        f"  ({path.stat().st_size / 1024**2:.2f} MB)"
    )

expected_best = (
    pipeline_dir
    / "yolo_runs"
    / "agent_selected_yolov8_global_wheat"
    / "weights"
    / "best.pt"
)

if expected_best.is_file():
    print("\nConfirmed target YOLO weights found：")
    print(expected_best)
else:
    print("\nWarning: best.pt was not found in the expected path, but all weights listed above will be packaged.")

# Generate file list
manifest_path = pipeline_dir / "_export_manifest.txt"
project_files = sorted(
    p for p in pipeline_dir.rglob("*")
    if p.is_file() and p != manifest_path
)

manifest_lines = [
    "Jiaozi Global Wheat generated pipeline",
    "",
    "WEIGHT FILES:",
]

manifest_lines.extend(
    f"{p.relative_to(pipeline_dir)}\t{p.stat().st_size}"
    for p in weight_files
)

manifest_lines.extend([
    "",
    "ALL FILES:",
])

manifest_lines.extend(
    f"{p.relative_to(pipeline_dir)}\t{p.stat().st_size}"
    for p in project_files
)

manifest_path.write_text(
    "\n".join(manifest_lines),
    encoding="utf-8",
)

total_size = sum(
    p.stat().st_size
    for p in pipeline_dir.rglob("*")
    if p.is_file()
)

print(f"\nTotal size before packaging：{total_size / 1024**2:.2f} MB")

archive_path = shutil.make_archive(
    "/content/jiaozi_generated_pipeline_bundle",
    "zip",
    root_dir=pipeline_dir.parent,
    base_dir=pipeline_dir.name,
)

print("The compressed file has been generated.：", archive_path)
print(f"compressed file size：{Path(archive_path).stat().st_size / 1024**2:.2f} MB")

files.download(archive_path)


找到以下权重文件：
  yolo_runs/agent_selected_yolov8_global_wheat/weights/best.pt  (49.70 MB)
  yolo_runs/agent_selected_yolov8_global_wheat/weights/last.pt  (49.70 MB)

确认找到目标 YOLO 权重：
/content/jiaozi_generated_pipeline/yolo_runs/agent_selected_yolov8_global_wheat/weights/best.pt

打包前总大小：715.63 MB
压缩包已生成： /content/jiaozi_generated_pipeline_bundle.zip
压缩包大小：701.59 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>